# Large Warehouse — All Pallet Sizes

Simulates a **1 500-aisle warehouse** (25 replicas × 60 aisle types) where every pallet-size
category has its own dedicated aisle type.  Inventory is stocked to **~85% bin utilisation**,
giving ≥ 100 000 placed items.

| Parameter | Value |
|-----------|-------|
| Aisle config types | 60 (48 pallet + 12 singleton) |
| Pallet sizes covered | small, medium, large, extra_large |
| Categories | food, clothing, electronic, furniture, seasonal, chemical |
| Handling types | conveyable, non-conveyable |
| Total aisles | 1 500 (25 replicas × 60 types) |
| Bin fill target | 85% |
| SKU catalog (≈85% of bins) | ~114 750 |
| Pickers | 25 |
| Batches | 1 000 |

Dedicated pallet-size aisles allow a per-size task-duration analysis that is not possible
when sizes are mixed within each aisle.

In [ ]:
import os
import random
import sys
import time as _time

_here = os.getcwd()
_warehouse_path = os.path.normpath(os.path.join(_here, '..', 'Warehouse'))
sys.path.insert(0, _warehouse_path)
sys.path.insert(0, _here)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde

from Aisle_Storage import Aisle
from Carton import Carton
from Inventory_Builder import Inventory_Builder, InventoryConfig
from Inventory_Management import Inventory_Manager
from Pick import PickConfig, PickSimulation
from Warehouse_Builder import AisleConfig, Warehouse_Builder, WarehouseConfig
from Workload_Builder import Batch, BatchConfig, Task

from Picking_Data import (
    BatchStats, TaskStats,
    create_run, init_run_db,
    load_recovered_params,
    save_batch_stats, load_batch_stats,
    save_task_stats,  load_task_stats,
)
from Simulation_Analytics import (
    build_placed_affinity,
    extract_batch_stats, extract_task_stats,
    flag_batch_outliers, flag_task_outliers,
    task_stats_to_aisle_loads, recover_params_to_db,
)
from Workload import WorkloadParams

print('Imports OK')

## Configuration

In [ ]:
SEED        = 42
N_BATCHES   = 1_000
K_PICKERS   = 25
DB_PATH     = os.path.join(_here, 'large_pallet_all_sizes.db')
_FILL_TARGET = 0.85

# ── Taxonomy ──────────────────────────────────────────────────────────────────
_CATEGORIES  = ['food', 'clothing', 'electronic', 'furniture', 'seasonal', 'chemical']
_ALL_SIZES   = ['small', 'medium', 'large', 'extra_large']

# Singleton bins use a mixed-size distribution (unchanged from existing design)
_CONV_SIZES  = ['small', 'medium', 'large']
_CONV_PROBS  = [0.25, 0.50, 0.25]
_NCONV_SIZES = ['medium', 'large', 'extra_large']
_NCONV_PROBS = [0.20, 0.50, 0.30]

# ── Aisle config types ────────────────────────────────────────────────────────
# Pallet aisles: one dedicated type per (handling × category × size)  → 48 types
# Singleton aisles: mixed-size distribution per (handling × category) → 12 types
# Total: 60 aisle config types
_AISLE_CFGS = []
for _size in _ALL_SIZES:
    for _cat in _CATEGORIES:
        _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'pallet', 10, 10, [_size], None))
        _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'pallet',  8, 10, [_size], None))
for _cat in _CATEGORIES:
    _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'singleton', 10, 10, _CONV_SIZES,  _CONV_PROBS))
    _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'singleton',  8, 10, _NCONV_SIZES, _NCONV_PROBS))

_N_TYPES      = len(_AISLE_CFGS)       # 60
_REPLICAS     = 25
_TOTAL_AISLES = _N_TYPES * _REPLICAS   # 1 500

WAREHOUSE_CFG = WarehouseConfig(
    total_aisles  = _TOTAL_AISLES,
    aisle_splits  = [1 / _N_TYPES] * _N_TYPES,
    aisle_configs = _AISLE_CFGS,
)

# Batch config — inventory_size patched after probe build below
_BATCH_MEAN_FRAC = 0.10
_BATCH_STD_FRAC  = 0.03

PICK_CFG = PickConfig(
    num_pickers      = K_PICKERS,
    x_move_time      = 1.0,
    y_move_time      = 0.5,
    pick_intercept   = 1.0,
    pick_weight_coef = 0.02,
    pick_volume_coef = 1e-4,
    cart_swap_coef   = 5.0,
)
WP = WorkloadParams.from_pick_config(PICK_CFG)

# Estimated bin counts
_est_conv  = sum(1 for c in _AISLE_CFGS if c.handling_type == 'conveyable')    * _REPLICAS * 10 * 10
_est_nconv = sum(1 for c in _AISLE_CFGS if c.handling_type == 'non-conveyable') * _REPLICAS *  8 * 10
_est_total = _est_conv + _est_nconv
print(f'Aisle config types  : {_N_TYPES}  (48 pallet + 12 singleton)')
print(f'Total aisles        : {_TOTAL_AISLES}  ({_REPLICAS} replicas × {_N_TYPES} types)')
print(f'Est. bins           : {_est_total:,}  (conv {_est_conv:,} + non-conv {_est_nconv:,})')
print(f'Est. items at 85%%  : {int(_est_total * _FILL_TARGET):,}')
print(f'Pickers             : {K_PICKERS}')
print(f'Batches             : {N_BATCHES:,}')

## Build World

In [ ]:
Carton.next_sku     = 1
Aisle.next_aisle_id = 1
random.seed(SEED)

t0 = _time.perf_counter()

# ── Probe to measure exact bin count ─────────────────────────────────────────
print('Probing warehouse for bin count...', end=' ', flush=True)
_probe     = Warehouse_Builder().from_config(WAREHOUSE_CFG).build()
TOTAL_BINS = len(_probe.bins)
N_SKUS     = int(_FILL_TARGET * TOTAL_BINS)
print(f'{TOTAL_BINS:,} bins  →  {N_SKUS:,} SKUs at {_FILL_TARGET:.0%} fill')
del _probe

BATCH_CFG = BatchConfig(
    inventory_size = N_SKUS,
    mean_fraction  = _BATCH_MEAN_FRAC,
    std_fraction   = _BATCH_STD_FRAC,
)

# ── Build warehouse (reset counters to match probe layout) ────────────────────
Carton.next_sku     = 1
Aisle.next_aisle_id = 1
random.seed(SEED)
print('Building warehouse...', end=' ', flush=True)
warehouse = Warehouse_Builder().from_config(WAREHOUSE_CFG).build()
print(f'{len(warehouse.aisles)} aisles, {len(warehouse.bins):,} bins')

# ── Build aisle metadata maps ─────────────────────────────────────────────────
AISLE_SIZE_MAP     = {a.aisle_id: a.storage_size  for a in warehouse.aisles}
AISLE_UNITTYPE_MAP = {a.aisle_id: a.unit_type     for a in warehouse.aisles}
AISLE_HANDLING_MAP = {a.aisle_id: a.handling_type for a in warehouse.aisles}

# ── Build inventory ───────────────────────────────────────────────────────────
INVENTORY_CFG = InventoryConfig(
    num_skus           = N_SKUS,
    handling_splits    = [0.5, 0.5],
    category_splits    = [1 / 6] * 6,
    singleton_fraction = 0.5,
)
print('Building inventory...', end=' ', flush=True)
inventory = Inventory_Builder().from_config(INVENTORY_CFG).build()
print(f'{len(inventory.cartons):,} cartons')

# ── Sparse pre-placement affinity (for batch sampling) ───────────────────────
# Groups SKUs by (handling, category); cap at 500/group → ~3 M entries.
# Used for lift-weighted Batch sampling; partner_map is cached on first call.
print('Building pre-placement affinity...', end=' ', flush=True)
_t_aff = _time.perf_counter()
AFFINITY = inventory.affinity_matrix(max_per_group=500)
print(f'{len(AFFINITY):,} entries  ({_time.perf_counter() - _t_aff:.2f}s)')

# ── Stock warehouse ───────────────────────────────────────────────────────────
print('Stocking warehouse...', end=' ', flush=True)
manager = Inventory_Manager(warehouse)
manager.enqueue_all(inventory.cartons, quantity=1)
placed = len(manager.unavailable)
print(f'{placed:,} bins filled  ({placed / TOTAL_BINS:.1%} utilisation)')

# ── Build placed affinity (for lift_sum in task stats) ───────────────────────
print('Building placed affinity...', end=' ', flush=True)
PLACED_AFFINITY = build_placed_affinity(warehouse, inventory, max_per_group=300)
n_pairs = len(PLACED_AFFINITY) // 2
print(f'{len(PLACED_AFFINITY):,} entries  ({n_pairs:,} unique pairs)')

print(f'World ready in {_time.perf_counter() - t0:.1f}s')

## Initialise Database

In [ ]:
init_run_db(DB_PATH)
RUN_ID = create_run(DB_PATH, 'large_pallet_all_sizes_1000')
print(f'DB  : {DB_PATH}')
print(f'Run : {RUN_ID}')

## Simulation Loop — 1 000 Batches

Each iteration:
1. Draw a batch using **lift-weighted affinity sampling** — SKUs that share a
   `(handling, category)` group are more likely to be co-selected
2. Decompose into per-aisle tasks
3. Run `PickSimulation` with 25 pickers
4. Extract `BatchStats` + `TaskStats` from events
5. Flush to DB every 100 batches

In [ ]:
_CHECKPOINT = 100

all_batch_stats: list[BatchStats] = []
all_task_stats:  list[TaskStats]  = []
_pending_batch: list[BatchStats]  = []
_pending_task:  list[TaskStats]   = []

t_loop  = _time.perf_counter()
skipped = 0

for _i in range(N_BATCHES):
    batch  = Batch(BATCH_CFG, inventory, affinity=AFFINITY)
    tasks  = Task.from_batch(batch, warehouse)

    if not tasks:
        skipped += 1
        continue

    events = PickSimulation(tasks, PICK_CFG).run()

    bs = extract_batch_stats(events, batch_id=_i, k_pickers=K_PICKERS, run_id=RUN_ID)
    ts = extract_task_stats(events, tasks, batch_id=_i,
                            affinity=PLACED_AFFINITY, wp=WP, run_id=RUN_ID)

    _pending_batch.append(bs)
    _pending_task.extend(ts)
    all_batch_stats.append(bs)
    all_task_stats.extend(ts)

    if len(_pending_batch) >= _CHECKPOINT:
        save_batch_stats(DB_PATH, RUN_ID, _pending_batch)
        save_task_stats( DB_PATH, RUN_ID, _pending_task)
        elapsed = _time.perf_counter() - t_loop
        rate    = (_i + 1) / elapsed
        print(f'Batch {_i+1:4d}/{N_BATCHES}  '
              f'tasks/batch={len(ts)}  '
              f'items/batch={bs.total_items}  '
              f'duration={bs.duration:.1f}  '
              f'{rate:.1f} batches/s',
              flush=True)
        _pending_batch.clear()
        _pending_task.clear()

if _pending_batch:
    save_batch_stats(DB_PATH, RUN_ID, _pending_batch)
    save_task_stats( DB_PATH, RUN_ID, _pending_task)

elapsed_total = _time.perf_counter() - t_loop
print(f'\nDone: {N_BATCHES - skipped} batches in {elapsed_total:.1f}s  '
      f'({skipped} skipped — no tasks)',
      flush=True)

## Load, Clean, and Summarise

In [ ]:
bs_raw = load_batch_stats(DB_PATH, RUN_ID)
ts_raw = load_task_stats( DB_PATH, RUN_ID)

bs_flagged = flag_batch_outliers(bs_raw, iqr_factor=1.5)
ts_flagged = flag_task_outliers( ts_raw, iqr_factor=1.5)

bs_clean = [s for s in bs_flagged if not s.is_outlier]
ts_clean = [s for s in ts_flagged if not s.is_outlier]

print(f'BatchStats : {len(bs_raw):>6,} total  |  {sum(s.is_outlier for s in bs_flagged):>5,} outliers  |  {len(bs_clean):>6,} clean')
print(f'TaskStats  : {len(ts_raw):>6,} total  |  {sum(s.is_outlier for s in ts_flagged):>5,} outliers  |  {len(ts_clean):>6,} clean')

## Summary Statistics Table

In [ ]:
def _batch_df(stats):
    return pd.DataFrame([
        {
            'batch_id'               : s.batch_id,
            'duration'               : s.duration,
            'num_tasks'              : s.num_tasks,
            'total_items'            : s.total_items,
            'completion_rate'        : s.total_items / s.duration if s.duration > 0 else 0,
            'avg_concurrent_pickers' : s.avg_concurrent_pickers,
            'picking_pct'            : s.picking_pct * 100,
            'traveling_pct'          : s.traveling_pct * 100,
            'is_outlier'             : s.is_outlier,
        }
        for s in stats
    ])

def _task_df(stats):
    return pd.DataFrame([
        {
            'batch_id'    : s.batch_id,
            'aisle_id'    : s.aisle_id,
            'picker_id'   : s.picker_id,
            'duration'    : s.duration,
            'W_a'         : s.W_a,
            'lift_sum'    : s.lift_sum,
            'num_bins'    : s.num_bins_visited,
            'total_items' : s.total_items,
            'is_outlier'  : s.is_outlier,
            'pallet_size' : AISLE_SIZE_MAP.get(s.aisle_id),
            'unit_type'   : AISLE_UNITTYPE_MAP.get(s.aisle_id),
            'handling'    : AISLE_HANDLING_MAP.get(s.aisle_id),
        }
        for s in stats
    ])

df_batch_all   = _batch_df(bs_flagged)
df_batch_clean = _batch_df(bs_clean)
df_task_all    = _task_df(ts_flagged)
df_task_clean  = _task_df(ts_clean)

_summary_cols = ['duration', 'num_tasks', 'total_items', 'completion_rate',
                 'avg_concurrent_pickers', 'picking_pct', 'traveling_pct']
summary = (
    df_batch_clean[_summary_cols]
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .round(3)
)
summary.index.name = 'stat'
print('=== Clean Batch Statistics (n =', len(bs_clean), ') ===')
display(summary)

_task_summary_cols = ['duration', 'W_a', 'num_bins', 'total_items']
task_summary = (
    df_task_clean[_task_summary_cols]
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .round(3)
)
task_summary.index.name = 'stat'
print('\n=== Clean Task Statistics (n =', len(ts_clean), ') ===')
display(task_summary)

## Plot 1 — Batch Completion Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
fig.suptitle('Batch Completion Time Distribution', fontsize=13, fontweight='bold')

for ax, data, label, color in [
    (axes[0], df_batch_all['duration'].values,   'All batches',   '#5b9bd5'),
    (axes[1], df_batch_clean['duration'].values,  'Clean batches', '#70ad47'),
]:
    if len(data) < 2:
        continue
    ax.hist(data, bins=40, color=color, alpha=0.65, edgecolor='white', label=label)
    kde = gaussian_kde(data, bw_method='silverman')
    xs  = np.linspace(data.min(), data.max(), 400)
    ax.plot(xs, kde(xs) * len(data) * (data.max() - data.min()) / 40,
            color=color, lw=2, alpha=0.9)
    mu  = float(np.mean(data))
    med = float(np.median(data))
    ax.axvline(mu,  color='#ff4444', lw=1.5, linestyle='--', label=f'Mean  {mu:.1f}')
    ax.axvline(med, color='#ff8800', lw=1.5, linestyle=':',  label=f'Median {med:.1f}')
    ax.set_xlabel('Batch duration  (simulation time units)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'{label}  (n={len(data):,})', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

## Plot 2 — Task Completion Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Task (Aisle) Completion Time Distribution', fontsize=13, fontweight='bold')

for ax, data, label, color in [
    (axes[0], df_task_all['duration'].values,   'All tasks',   '#5b9bd5'),
    (axes[1], df_task_clean['duration'].values,  'Clean tasks', '#70ad47'),
]:
    if len(data) < 2:
        continue
    ax.hist(data, bins=50, color=color, alpha=0.65, edgecolor='white', label=label)
    kde = gaussian_kde(data, bw_method='silverman')
    xs  = np.linspace(data.min(), data.max(), 400)
    ax.plot(xs, kde(xs) * len(data) * (data.max() - data.min()) / 50,
            color=color, lw=2, alpha=0.9)
    mu  = float(np.mean(data))
    med = float(np.median(data))
    ax.axvline(mu,  color='#ff4444', lw=1.5, linestyle='--', label=f'Mean  {mu:.1f}')
    ax.axvline(med, color='#ff8800', lw=1.5, linestyle=':',  label=f'Median {med:.1f}')
    ax.set_xlabel('Task duration  (simulation time units)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'{label}  (n={len(data):,})', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

## Plot 3 — Batch Completion Rate (Rolling Average)

In [ ]:
_WINDOW = 50

df_sorted = df_batch_clean.sort_values('batch_id').reset_index(drop=True)
df_sorted['rate_roll'] = df_sorted['completion_rate'].rolling(_WINDOW, min_periods=1).mean()
df_sorted['dur_roll']  = df_sorted['duration'].rolling(_WINDOW, min_periods=1).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle(f'Batch Completion Rate  (rolling {_WINDOW}-batch window, clean data)',
             fontsize=13, fontweight='bold')

ax1.scatter(df_sorted['batch_id'], df_sorted['completion_rate'],
            s=5, alpha=0.25, color='#5b9bd5', label='Individual batches')
ax1.plot(df_sorted['batch_id'], df_sorted['rate_roll'],
         color='#1f4e79', lw=2, label=f'Rolling mean ({_WINDOW})')
ax1.set_ylabel('Items / time unit', fontsize=10)
ax1.set_title('Throughput rate  (items picked per time unit)', fontsize=10)
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

ax2.scatter(df_sorted['batch_id'], df_sorted['duration'],
            s=5, alpha=0.25, color='#70ad47', label='Individual batches')
ax2.plot(df_sorted['batch_id'], df_sorted['dur_roll'],
         color='#375623', lw=2, label=f'Rolling mean ({_WINDOW})')
ax2.set_xlabel('Batch ID', fontsize=10)
ax2.set_ylabel('Duration  (time units)', fontsize=10)
ax2.set_title('Batch completion time', fontsize=10)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## Plot 4 — Picker Concurrency Distribution

In [ ]:
conc = df_batch_clean['avg_concurrent_pickers'].values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle(f'Picker Concurrency Distribution  (n={len(conc):,} clean batches)',
             fontsize=13, fontweight='bold')

ax1.hist(conc, bins=40, color='#9dc3e6', alpha=0.75, edgecolor='white')
if len(conc) > 1:
    kde = gaussian_kde(conc, bw_method='silverman')
    xs  = np.linspace(conc.min(), conc.max(), 400)
    ax1.plot(xs, kde(xs) * len(conc) * (conc.max() - conc.min()) / 40,
             color='#2e75b6', lw=2)
ax1.axvline(float(np.mean(conc)),   color='#ff4444', lw=1.5, linestyle='--',
            label=f'Mean  {np.mean(conc):.2f}')
ax1.axvline(float(np.median(conc)), color='#ff8800', lw=1.5, linestyle=':',
            label=f'Median {np.median(conc):.2f}')
ax1.axvline(K_PICKERS, color='#a9a9a9', lw=1.0, linestyle='-.',
            label=f'Max pickers ({K_PICKERS})')
ax1.set_xlabel('Avg concurrent pickers  (picking state only)', fontsize=10)
ax1.set_ylabel('Count', fontsize=10)
ax1.set_title('Histogram + KDE', fontsize=10)
ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)

xs_sorted = np.sort(conc)
cdf = np.arange(1, len(xs_sorted) + 1) / len(xs_sorted)
ax2.plot(xs_sorted, cdf, color='#2e75b6', lw=2)
ax2.axvline(float(np.mean(conc)),              color='#ff4444', lw=1.5, linestyle='--',
            label=f'Mean  {np.mean(conc):.2f}')
ax2.axvline(float(np.percentile(conc, 50)),    color='#ff8800', lw=1.5, linestyle=':',
            label=f'p50 {np.percentile(conc, 50):.2f}')
ax2.axvline(float(np.percentile(conc, 90)),    color='#7030a0', lw=1.5, linestyle=':',
            label=f'p90 {np.percentile(conc, 90):.2f}')
ax2.set_xlabel('Avg concurrent pickers', fontsize=10)
ax2.set_ylabel('Cumulative probability', fontsize=10)
ax2.set_title('CDF', fontsize=10)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

fig.tight_layout()
plt.show()

## Plot 5 — Picker Utilisation Breakdown

In [ ]:
pick_pct = df_batch_clean['picking_pct'].values
trav_pct = df_batch_clean['traveling_pct'].values

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Picker Utilisation Breakdown  (clean batches)', fontsize=13, fontweight='bold')

bp = axes[0].boxplot(
    [pick_pct, trav_pct],
    labels=['Picking %', 'Traveling %'],
    patch_artist=True,
    medianprops=dict(color='black', lw=2),
)
for patch, color in zip(bp['boxes'], ['#70ad47', '#5b9bd5']):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[0].set_ylabel('Fraction of aggregate picker time (%)', fontsize=9)
axes[0].set_title('Box plots', fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(pick_pct, bins=35, color='#70ad47', alpha=0.75, edgecolor='white')
axes[1].axvline(float(np.mean(pick_pct)), color='#375623', lw=2,
                label=f'Mean {np.mean(pick_pct):.1f}%')
axes[1].set_xlabel('Picking %', fontsize=10)
axes[1].set_ylabel('Count', fontsize=10)
axes[1].set_title('Distribution of picking %', fontsize=10)
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

means = [float(np.mean(pick_pct)), float(np.mean(trav_pct))]
axes[2].barh(['Utilisation'], [means[0]], color='#70ad47', alpha=0.85,
             label=f'Picking {means[0]:.1f}%')
axes[2].barh(['Utilisation'], [means[1]], left=[means[0]], color='#5b9bd5', alpha=0.85,
             label=f'Traveling {means[1]:.1f}%')
axes[2].set_xlim(0, 100)
axes[2].set_xlabel('Mean fraction of picker time (%)', fontsize=10)
axes[2].set_title('Aggregate mean split', fontsize=10)
axes[2].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
axes[2].legend(fontsize=9, loc='lower right')
axes[2].grid(axis='x', alpha=0.3)

fig.tight_layout()
plt.show()

## Plot 6 — Pallet Size Analysis

Because every pallet aisle is dedicated to a single size, we can compare task duration,
items per task, and visit frequency broken out by pallet size.  Singleton aisles (mixed sizes)
are shown separately for reference.

In [ ]:
_SIZE_ORDER   = ['small', 'medium', 'large', 'extra_large']
_SIZE_COLORS  = ['#9dc3e6', '#5b9bd5', '#2e75b6', '#1f4e79']
_SIZE_LABELS  = ['Small', 'Medium', 'Large', 'Extra-Large']

# ── Pallet-only clean tasks (unit_type == 'pallet', pallet_size is not None) ──
df_pallet = df_task_clean[
    (df_task_clean['unit_type'] == 'pallet') &
    (df_task_clean['pallet_size'].notna())
].copy()

df_pallet['pallet_size'] = pd.Categorical(
    df_pallet['pallet_size'], categories=_SIZE_ORDER, ordered=True
)

# ── Subplot 1: Task duration box-plots per size ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Pallet Aisle Analysis by Storage Size  (clean tasks only)',
             fontsize=13, fontweight='bold')

# Box plots
size_groups = [df_pallet[df_pallet['pallet_size'] == s]['duration'].values
               for s in _SIZE_ORDER]
bp = axes[0].boxplot(
    size_groups, labels=_SIZE_LABELS,
    patch_artist=True, medianprops=dict(color='black', lw=2),
)
for patch, color in zip(bp['boxes'], _SIZE_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[0].set_ylabel('Task duration  (time units)', fontsize=10)
axes[0].set_title('Task duration distribution per pallet size', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Mean items-per-task per size
mean_items = [df_pallet[df_pallet['pallet_size'] == s]['total_items'].mean()
              for s in _SIZE_ORDER]
bars = axes[1].bar(_SIZE_LABELS, mean_items, color=_SIZE_COLORS, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, mean_items):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9)
axes[1].set_ylabel('Mean items per task', fontsize=10)
axes[1].set_title('Mean items per aisle task per pallet size', fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

# Task visit counts per size
visit_counts = [len(df_pallet[df_pallet['pallet_size'] == s]) for s in _SIZE_ORDER]
bars2 = axes[2].bar(_SIZE_LABELS, visit_counts, color=_SIZE_COLORS, alpha=0.85, edgecolor='white')
for bar, val in zip(bars2, visit_counts):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)
axes[2].set_ylabel('Total task visits across all batches', fontsize=10)
axes[2].set_title('Task visit frequency per pallet size', fontsize=10)
axes[2].grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

# ── Overlaid KDE: task duration by size ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))
fig.suptitle('Task Duration KDE — Pallet Sizes Overlaid  (clean tasks)',
             fontsize=12, fontweight='bold')

for size, color, label in zip(_SIZE_ORDER, _SIZE_COLORS, _SIZE_LABELS):
    vals = df_pallet[df_pallet['pallet_size'] == size]['duration'].values
    if len(vals) > 1:
        kde = gaussian_kde(vals, bw_method='silverman')
        xs  = np.linspace(vals.min(), vals.max(), 400)
        ax.plot(xs, kde(xs), color=color, lw=2.5,
                label=f'{label}  μ={vals.mean():.1f}  n={len(vals):,}')
        ax.axvline(vals.mean(), color=color, lw=1.0, linestyle='--', alpha=0.6)

ax.set_xlabel('Task duration  (time units)', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# ── Conveyable vs Non-conveyable breakdown ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Task Duration — Pallet Size × Handling Type  (mean, clean tasks)',
             fontsize=12, fontweight='bold')

for ax, handling, color_set in [
    (axes[0], 'conveyable',     ['#c6e0f5', '#9dc3e6', '#5b9bd5', '#2e75b6']),
    (axes[1], 'non-conveyable', ['#fce4d6', '#f4b183', '#e36c09', '#843c0c']),
]:
    df_h = df_pallet[df_pallet['handling'] == handling]
    means = [df_h[df_h['pallet_size'] == s]['duration'].mean() for s in _SIZE_ORDER]
    bars  = ax.bar(_SIZE_LABELS, means, color=color_set, alpha=0.9, edgecolor='white')
    for bar, val in zip(bars, means):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'{handling.capitalize()} pallet aisles', fontsize=10)
    ax.set_ylabel('Mean task duration  (time units)', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

print('\n=== Mean task duration by pallet size (all handling) ===')
print(df_pallet.groupby('pallet_size', observed=True)['duration']
      .agg(['mean', 'median', 'std', 'count']).round(3))

## Per-Aisle Heatmap (Duration vs Aisle ID)

In [ ]:
aisle_dur = (
    df_task_clean
    .groupby('aisle_id')['duration']
    .agg(['mean', 'median', 'count'])
    .rename(columns={'mean': 'mean_dur', 'median': 'median_dur', 'count': 'n_picks'})
    .reset_index()
    .sort_values('aisle_id')
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 7), sharex=True)
fig.suptitle('Per-Aisle Task Duration Summary  (clean tasks, 1 500 aisles)',
             fontsize=13, fontweight='bold')

ax1.bar(aisle_dur['aisle_id'], aisle_dur['mean_dur'],
        color='#5b9bd5', alpha=0.7, width=1.0, label='Mean duration')
ax1.plot(aisle_dur['aisle_id'], aisle_dur['median_dur'],
         color='#ff4444', lw=1.0, linestyle='--', label='Median duration')
ax1.set_ylabel('Duration  (time units)', fontsize=10)
ax1.set_title('Mean and median task duration per aisle', fontsize=10)
ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)

ax2.bar(aisle_dur['aisle_id'], aisle_dur['n_picks'],
        color='#70ad47', alpha=0.7, width=1.0)
ax2.set_xlabel('Aisle ID', fontsize=10)
ax2.set_ylabel('Pick count', fontsize=10)
ax2.set_title('Total task visits per aisle across all batches', fontsize=10)
ax2.grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

## Parameter Recovery — L_a = W_a + λ*(W_a/k)^γ · SUM_lift

Convert `TaskStats` into `AisleLoadRecord` objects and recover `LoadParams` (λ, γ) via
log-linear OLS after IQR outlier cleaning.  Results are written to the **`aisle_loads`** and
**`recovered_params`** tables in the SQLite database and exported to `recovered_params.json`.

In [ ]:
JSON_PATH = os.path.join(_here, 'recovered_params.json')

aisle_load_records = task_stats_to_aisle_loads(all_task_stats, run_id=RUN_ID)

n_with_lift = sum(1 for r in aisle_load_records if r.lift_sum > 0)
print(f'AisleLoadRecords  : {len(aisle_load_records):,} total')
print(f'  lift_sum > 0    : {n_with_lift:,}  ({n_with_lift / max(len(aisle_load_records), 1):.1%})')
print(f'  lift_sum = 0    : {len(aisle_load_records) - n_with_lift:,}  (excluded from OLS fit)')
print()

print('=== Recovery pipeline ===')
rp = recover_params_to_db(
    db_path    = DB_PATH,
    run_id     = RUN_ID,
    records    = aisle_load_records,
    k_per_task = 1,
    json_path  = JSON_PATH,
    do_plot    = False,
)

In [ ]:
rp_loaded = load_recovered_params(DB_PATH, RUN_ID)

if rp_loaded is None:
    print('No recovered parameters found.')
else:
    print('=== Recovered Load Parameters ===')
    print(f'  λ  (lambda)  : {rp_loaded.lambda_:.6f}')
    print(f'  γ  (gamma)   : {rp_loaded.gamma:.6f}')
    print(f'  k  (pickers) : {rp_loaded.k:.0f}')
    print()
    print('=== Fit Quality ===')
    print(f'  RMSE (raw fit)   : {rp_loaded.rmse_raw:.4f}')
    print(f'  RMSE (clean fit) : {rp_loaded.rmse_clean:.4f}')
    print(f'  Improvement      : {(rp_loaded.rmse_raw - rp_loaded.rmse_clean) / rp_loaded.rmse_raw * 100:.1f}%')
    print()
    print('=== Coverage ===')
    print(f'  Observations (total) : {rp_loaded.n_samples:,}')
    print(f'  Observations (clean) : {rp_loaded.n_clean:,}')
    n_out = rp_loaded.n_samples - rp_loaded.n_clean
    print(f'  Outliers removed     : {n_out:,}  ({n_out / max(rp_loaded.n_samples, 1):.1%})')
    print(f'  Recovered at         : {rp_loaded.timestamp}')
    if os.path.exists(JSON_PATH):
        print(f'  JSON export          : {JSON_PATH}')

    al_records = [r for r in aisle_load_records if r.lift_sum > 0 and not r.is_outlier]
    if al_records:
        W     = np.array([r.W_a          for r in al_records])
        S     = np.array([r.lift_sum     for r in al_records])
        L     = np.array([r.observed_L_a for r in al_records])
        k     = rp_loaded.k
        lm    = rp_loaded.lambda_
        gm    = rp_loaded.gamma
        L_pred = W + lm * (W / k) ** gm * S

        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        fig.suptitle('Parameter Recovery — Observed vs Predicted L_a',
                     fontsize=13, fontweight='bold')

        ax = axes[0]
        ax.scatter(L_pred, L, s=4, alpha=0.2, color='#5b9bd5')
        lims = [min(L_pred.min(), L.min()), max(L_pred.max(), L.max())]
        ax.plot(lims, lims, color='#ff4444', lw=1.5, linestyle='--', label='y = x')
        ax.set_xlabel('Predicted L_a', fontsize=10)
        ax.set_ylabel('Observed L_a', fontsize=10)
        ax.set_title(f'Predicted vs Observed  (n={len(L):,} clean)', fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

        ax = axes[1]
        resid = L - L_pred
        ax.hist(resid, bins=50, color='#70ad47', alpha=0.75, edgecolor='white')
        ax.axvline(0, color='#ff4444', lw=1.5, linestyle='--', label='zero residual')
        ax.axvline(float(np.mean(resid)), color='#ff8800', lw=1.5, linestyle=':',
                   label=f'Mean {np.mean(resid):.3f}')
        ax.set_xlabel('Residual  (observed − predicted)', fontsize=10)
        ax.set_ylabel('Count', fontsize=10)
        ax.set_title(f'Residual distribution  RMSE={rp_loaded.rmse_clean:.4f}', fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

        fig.tight_layout()
        plt.show()